# Real Estate Price Prediction Model

We are using "property_data.csv" dataset from kaggle : https://www.kaggle.com/datasets/hassaanmustafavi/pakistan-real-estate-property-listings-dataset

We are gonna make a model that can predict the price of the property.

---

# ***Data Analysis & Preprocessing***

---

## Loading Dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/property_data.csv")
df.head()

,index,url,type,purpose,area,bedroom,bath,added,price,location,location_city
0,0,https://www.zameen.com/Property/dha_defence_dh...,House,For Sale,1 Kanal,7,6,4 days ago,PKR\n19 Crore,DHA Defence,Islamabad
1,1,https://www.zameen.com/Property/g_15_g-15_1_in...,House,For Sale,14.2 Marla,6,6,4 days ago,PKR\n6 Crore,G-15,Islamabad
2,2,https://www.zameen.com/Property/islamabad_g-16...,House,For Sale,1 Kanal,8,7,4 days ago,PKR\n7 Crore,G-16,Islamabad
3,3,https://www.zameen.com/Property/b_17_mpchs_-_m...,House,For Sale,8 Marla,4,6,4 days ago,PKR\n2.65 Crore,B-17,Islamabad
4,4,https://www.zameen.com/Property/islamabad_isla...,Flat,For Sale,2.4 Marla,1,1,4 days ago,PKR\n40 Lakh,Islamabad - Murree Expressway,Islamabad


In [3]:
df.shape

(116033, 11)

OK...so there are some columns that we definitely don't need, at all. Like index, url, added.

---

### Dropping unnecessary columns

In [4]:
df_v2 = df.drop(columns = ["index", "url", "added"]).copy()
df_v2.head(2)

,type,purpose,area,bedroom,bath,price,location,location_city
0,House,For Sale,1 Kanal,7,6,PKR\n19 Crore,DHA Defence,Islamabad
1,House,For Sale,14.2 Marla,6,6,PKR\n6 Crore,G-15,Islamabad


---

### Analyzing other columns

Ok so, there is a column named purpose. Let's take a look at it.

In [5]:
print(df_v2.shape)
print(f'Total Unique Values in the "purpose" col : {df_v2["purpose"].unique().size}')
df_v2["purpose"].value_counts()

(116033, 8)
Total Unique Values in the "purpose" col : 154


purpose
For Sale                                                                                                                                      87451
For Rent                                                                                                                                      28297
For Buy                                                                                                                                         117
For Sale Block A                                                                                                                                  3
For Rent At Ghauri Garden Lathrar Road                                                                                                            3
                                                                                                                                              ...  
For Rent Daimond City                                                                                   

There are different "purposes" of properties. But we don't need them all. ***As we are making a model that can predict the prices of the properties that are for sale, so we only need the ones that are for sale. let's seperate them.*** 

---

### Seperating the lands that are for sale


I am gonna seperate the lands that are for sale using the keyword "Sale".

In [6]:
df_v3 = df_v2[df_v2["purpose"].str.contains("Sale", case=False, na=False)].copy()
# case = False removes the case sensitivity check or whatever you wanna call it. Basically it will just for all the sale words, doesn't matter if they are in capital letters or small. 
# na = False ignores all the NaN values.
df_v3.head(2)

,type,purpose,area,bedroom,bath,price,location,location_city
0,House,For Sale,1 Kanal,7,6,PKR\n19 Crore,DHA Defence,Islamabad
1,House,For Sale,14.2 Marla,6,6,PKR\n6 Crore,G-15,Islamabad


In [7]:
print(df_v3.shape)
print(df_v3["purpose"].value_counts())

(87567, 8)
purpose
For Sale                                                      87451
For Sale Block A                                                  3
For Sale On 3 Years                                               2
For Sale At Prime Location                                        2
For Sale Tulip Block                                              2
                                                              ...  
For Sale Or                                                       1
For Sale I.I.Chundrigar Road                                      1
For Sale At Main Church Road Near Haji Pura Police Station        1
For Sale On Easy                                                  1
For Sale Life Time Commerical Paid                                1
Name: count, Length: 110, dtype: int64


We don't need the purpose column anymore. Let's drop it too.

---

### Dropping the "purpose" column

In [8]:
df_v3 = df_v3.drop(columns= "purpose")
df_v3["price"] = df_v3.pop("price") #puts the price(target) column at the end
df_v3.head(1)

,type,area,bedroom,bath,location,location_city,price
0,House,1 Kanal,7,6,DHA Defence,Islamabad,PKR\n19 Crore


---

### Remaining Columns

Let's check the remaining columns.

---

**"type"**

In [9]:
print(f'Null Values in the "type" col : {df_v3["type"].isna().sum()}')
print(f'Unique Values in the "type" col :\n {df_v3["type"].unique()}')
print(df_v3["type"].value_counts())

Null Values in the "type" col : 0
Unique Values in the "type" col :
 <StringArray>
[            'House',              'Flat',        'Farm House',
     'Upper Portion',     'Lower Portion',         'Penthouse',
              'Room',  'Residential Plot',   'Commercial Plot',
         'Plot File',   'Industrial Land', 'Agricultural Land',
         'Plot Form',              'Shop',             'Other',
            'Office',          'Building',           'Factory',
         'Warehouse',        'Commercial',        'Pent House',
         'Apartment',       'Guest House',      'Hotel Suites',
    'Farmhouse Plot',             'Plaza',              'Hall',
              'Land',        'Food Court']
Length: 29, dtype: str
type
Residential Plot     38365
House                28305
Flat                  7115
Shop                  4166
Commercial Plot       1875
Building              1786
Upper Portion         1221
Plot File             1030
Agricultural Land      978
Office                 962


Ok so there is a duplicate column : Pent House(48) and Penthouse(20). And Farm House and Farm House Plot. Let's fix it.

In [10]:
df_v3["type"] = df_v3["type"].replace(
    {
        "Pent House" : "Penthouse",
        'Farmhouse Plot': 'Farm House'
    }
)

In [11]:
print(f'Unique Values in the "type" col : {df_v3["type"].unique().size}')
# print(df_v3["type"].value_counts())

Unique Values in the "type" col : 27


Now, we have 27 unique values instead of 29.

To make it more manageable, let's name all the properties under 500 value count to "Other".

In [12]:
rare = df_v3["type"].value_counts()[df_v3["type"].value_counts() < 500].index
df_v3["type"] = df_v3["type"].replace(rare, "Other")

In [13]:
print(f'Unique Values in the "type" col : {df_v3["type"].unique().size}')
print(df_v3["type"].value_counts())

Unique Values in the "type" col : 11
type
Residential Plot     38365
House                28305
Flat                  7115
Shop                  4166
Commercial Plot       1875
Building              1786
Other                 1764
Upper Portion         1221
Plot File             1030
Agricultural Land      978
Office                 962
Name: count, dtype: int64


This looks much cleaner.

---

**"bedroom"**

In [14]:
print(f'NaN Values in the "bedroom" col : {df_v3["bedroom"].isna().sum()}')
print(f'Unique Values in the "bedroom" col : {df_v3["bedroom"].unique()}')
# df_v3["bedroom"].value_counts()

NaN Values in the "bedroom" col : 0
Unique Values in the "bedroom" col : <StringArray>
[     '7',      '6',      '8',      '4',      '1',      '3',      '2',
      '5',      '9',      '-',     '10',     '11',     '12',     '13',
    '200',     '20',     '16',    '100',     '15',    '10+', 'Studio',
    '2.0',    '3.0',    '4.0',    '7.0',    '5.0',    '6.0',    '8.0',
    '1.0',      '0']
Length: 30, dtype: str


Ok...200 bedrooms? damn. LOL

In [15]:
df_v3[df_v3["bedroom"].isin(["100", "200"])]

,type,area,bedroom,bath,location,location_city,price
34652,Other,8 Kanal,200,-,Changla Gali,Galyat,PKR\n75 Crore
39816,Other,3.5 Kanal,100,-,Changla Gali,Galyat,PKR\n70 Crore


Well, a 8 kanal house can't have 200 bedrooms...unless it's Burj Khalifa built on a 8 Kanal area.
We also need to do something  about the studio and the 10+ values.

In [16]:
df_v3["bedroom"] = df_v3["bedroom"].replace(
    {"200" : np.nan,
     "100" : np.nan,
     "Studio" : np.nan,
     "10+" : 10}
)

In [17]:
print(f'Unique Values in the "bedroom" col : {df_v3["bedroom"].unique()}')
print(f'NaN Values in the "bedroom" col : {df_v3["bedroom"].isna().sum()}')

Unique Values in the "bedroom" col : ['7' '6' '8' '4' '1' '3' '2' '5' '9' '-' '10' '11' '12' '13' nan '20' '16'
 '15' 10 '2.0' '3.0' '4.0' '7.0' '5.0' '6.0' '8.0' '1.0' '0']
NaN Values in the "bedroom" col : 46


In [18]:
df_v3["type"][df_v3["bedroom"] == "-"].unique()

<StringArray>
[             'Flat',             'House',             'Other',
     'Upper Portion',  'Residential Plot',   'Commercial Plot',
         'Plot File', 'Agricultural Land',              'Shop',
            'Office',          'Building']
Length: 11, dtype: str

I am gonna take the median of the other bedroom values and replace them for all these plot values except for "Plot File", "Agricultural Land", and "Shop".

But I need to convert the rest to numbers first.

In [19]:
df_v3["bedroom"] = pd.to_numeric(df_v3["bedroom"], errors="coerce") # coerce converts all the non-convertable values to NaN

In [20]:
print(f'Unique Values in the "bedroom" col : {df_v3["bedroom"].unique()}')
print(f'NaN Values in the "bedroom" col : {df_v3["bedroom"].isna().sum()}')

Unique Values in the "bedroom" col : [ 7.  6.  8.  4.  1.  3.  2.  5.  9. nan 10. 11. 12. 13. 20. 16. 15.  0.]
NaN Values in the "bedroom" col : 24123


In [21]:
df_v3["type"][df_v3["bedroom"].isna()].unique()

<StringArray>
[             'Flat',             'House',             'Other',
     'Upper Portion',  'Residential Plot',   'Commercial Plot',
         'Plot File', 'Agricultural Land',              'Shop',
            'Office',          'Building']
Length: 11, dtype: str

In [22]:
# check median per type first
df_v3[df_v3["bedroom"].notna()].groupby("type")["bedroom"].median()

type
Agricultural Land    0.0
Building             6.0
Commercial Plot      0.0
Flat                 2.0
House                5.0
Office               1.0
Other                2.0
Plot File            0.0
Residential Plot     0.0
Shop                 0.0
Upper Portion        3.0
Name: bedroom, dtype: float64

In [23]:
bedroom_medians = {
    "Agricultural Land": 0, 
    "Building": 6,
    "Commercial Plot": 0,   
    "Flat": 2,
    "House": 5,
    "Office": 0,            
    "Other": 2,
    "Plot File": 0,         
    "Residential Plot": 0,  
    "Shop": 0,              
    "Upper Portion": 3,
}

for property_type, median_val in bedroom_medians.items():
    mask = (df_v3["bedroom"].isna()) & (df_v3["type"] == property_type)
    df_v3.loc[mask, "bedroom"] = median_val

In [24]:
print(f'Unique Values in the "bedroom" col : {df_v3["bedroom"].unique()}')
print(f'NaN Values in the "bedroom" col : {df_v3["bedroom"].isna().sum()}')

Unique Values in the "bedroom" col : [ 7.  6.  8.  4.  1.  3.  2.  5.  9. 10. 11. 12. 13.  0. 20. 16. 15.]
NaN Values in the "bedroom" col : 0


Yehaaaa!!

---

**"bath"**

In [25]:
print(f'Null Values in the "bath" col : {df_v3["bath"].isna().sum()}')
print(f'Total Unique Values in the "bath" col : {df_v3["bath"].unique()}')
df_v3["bath"].value_counts()

Null Values in the "bath" col : 0
Total Unique Values in the "bath" col : <StringArray>
[  '6',   '7',   '1',   '2',   '4',   '3',   '5',   '-',  '10',   '8',   '9',
 '10+', '2.0', '3.0', '4.0', '7.0', '5.0', '6.0', '8.0', '1.0',   '0']
Length: 21, dtype: str


bath
0      26392
-      23789
6       8375
5       7119
4       6648
2       4919
3       4386
1       2021
2.0     1117
3.0     1101
7        856
8        289
5.0      202
4.0      115
7.0       79
10        40
9         38
10+       37
6.0       35
1.0        5
8.0        4
Name: count, dtype: int64

We are gonna replace the 10+ values with 10.

In [26]:
df_v3["bath"] = df_v3["bath"].replace("10+", 10)

In [27]:
print(f'Total Unique Values in the "bath" col : {df_v3["bath"].unique()}')

Total Unique Values in the "bath" col : ['6' '7' '1' '2' '4' '3' '5' '-' '10' '8' '9' 10 '2.0' '3.0' '4.0' '7.0'
 '5.0' '6.0' '8.0' '1.0' '0']


Gotta do something about the "-" values. I am thinking to go with the similar method as I did with bedroom col.

Let's convert the rest to numbers first.

In [28]:
df_v3["bath"] = pd.to_numeric(df_v3["bath"], errors="coerce")

In [29]:
print(f'Total Unique Values in the "bath" col : {df_v3["bath"].unique()}')

Total Unique Values in the "bath" col : [ 6.  7.  1.  2.  4.  3.  5. nan 10.  8.  9.  0.]


In [30]:
df_v3[df_v3["bath"].notna()].groupby("type")["bath"].median()

type
Agricultural Land    0.0
Building             6.0
Commercial Plot      0.0
Flat                 2.0
House                5.0
Office               1.0
Other                2.0
Plot File            0.0
Residential Plot     0.0
Shop                 0.0
Upper Portion        3.0
Name: bath, dtype: float64

In [31]:
bath_medians = {
    "Agricultural Land": 0, 
    "Building": 6,
    "Commercial Plot": 0,   
    "Flat": 2,
    "House": 5,
    "Office": 1,            
    "Other": 2,
    "Plot File": 0,         
    "Residential Plot": 0,  
    "Shop": 0,              
    "Upper Portion": 3,
}

for property_type, median_val in bath_medians.items():
    mask = (df_v3["bath"].isna()) & (df_v3["type"] == property_type)
    df_v3.loc[mask, "bath"] = median_val

In [32]:
print(f'Total Unique Values in the "bath" col : {df_v3["bath"].unique()}')

Total Unique Values in the "bath" col : [ 6.  7.  1.  2.  4.  3.  5. 10.  8.  9.  0.]


---

**"location"**

In [33]:
print(f'Null Values in the "location" col : {df_v3["location"].isna().sum()}')
print(f'Total Unique Values in the "location" col : {df_v3["location"].unique().size}')
df_v3["location"].value_counts()

Null Values in the "location" col : 22
Total Unique Values in the "location" col : 3002


location
DHA Defence               5881
Bahria Town Phase 8       4389
Bahria Town Rawalpindi    2070
DHA Phase 6               2056
Others                    1939
                          ... 
Kahuta Triangle              1
Phimbal Road Daultala        1
Ghanta Ghar Chowk            1
Gulistan Enclave Wah         1
Kohistan Enclave             1
Name: count, Length: 3001, dtype: int64

The location column has 3002 unique values. Encoding this many categories would either explode the feature space with OHE or produce unreliable target encodings for rare locations. Since location_city captures the geographical signal with a manageable unique values, we are gonna drop location.

In [34]:
df_v3 = df_v3.drop(columns="location")
df_v3.head(2)

,type,area,bedroom,bath,location_city,price
0,House,1 Kanal,7.0,6.0,Islamabad,PKR\n19 Crore
1,House,14.2 Marla,6.0,6.0,Islamabad,PKR\n6 Crore


---

"**location_city**"

In [35]:
print(f'Null Values in the "location_city" col : {df_v3["location_city"].isna().sum()}')
print(f'Total Unique Values in the "location_city" col : {df_v3["location_city"].unique()}')
df_v3["location_city"].value_counts()

Null Values in the "location_city" col : 0
Total Unique Values in the "location_city" col : <StringArray>
[    ' Islamabad',       ' Karachi',        ' Lahore',    ' Rawalpindi',
    ' Abbottabad',        ' Punjab', ' Ahmedpur East',        ' Alipur',
      ' Arifwala',        ' Astore',
 ...
    'Sheikhupura',          'Hunza',         'Jhelum',          'Kasur',
        'DI Khan',         'Gilgit',        'Khanpur',         'Mardan',
   'Muzaffarabad',     'Gujar Khan']
Length: 242, dtype: str


location_city
Lahore          13189
Islamabad       11521
Rawalpindi      10348
Karachi          6189
Quetta           4709
                ...  
DI Khan             1
Gilgit              1
Khanpur             1
Muzaffarabad        1
Gujar Khan          1
Name: count, Length: 242, dtype: int64

The names of cities needs to be stripped.

In [36]:
df_v3["location_city"] = df_v3["location_city"].str.strip()

In [37]:
print(f'Total Unique Values in the "location_city" col : {df_v3["location_city"].unique()}')

Total Unique Values in the "location_city" col : <StringArray>
[     'Islamabad',        'Karachi',         'Lahore',     'Rawalpindi',
     'Abbottabad',         'Punjab',  'Ahmedpur East',         'Alipur',
       'Arifwala',         'Astore',
 ...
       'Sudhnoti',        'Sujawal',     'Tando Bago',     'Waziristan',
      'Upper Dir',     'Qazi Ahmed',        'Loralai',         'Sukkur',
 'Shahpur Chakar',        'DI Khan']
Length: 207, dtype: str


Let's try to make the locations more manageable.

In [38]:
df_v3["location_city"].value_counts()[df_v3["location_city"].value_counts() < 300]

location_city
Sindh             249
Sheikhupura       205
Gujrat            187
Mardan            186
Murree            158
                 ... 
Mitha Tiwana        1
Qazi Ahmed          1
Loralai             1
Shahpur Chakar      1
DI Khan             1
Name: count, Length: 188, dtype: int64

Okay, so there are 188 locations that have values less than 100. Let's put them under the title "Other".

In [39]:
rare_cities = df_v3["location_city"].value_counts()[df_v3["location_city"].value_counts() < 500].index
df_v3["location_city"] = df_v3["location_city"].replace(rare_cities, "Other")

In [40]:
print(f'Total Unique Values in the "location_city" col : {df_v3["location_city"].unique()}')

Total Unique Values in the "location_city" col : <StringArray>
[ 'Islamabad',    'Karachi',     'Lahore', 'Rawalpindi',      'Other',
     'Punjab', 'Bahawalpur', 'Faisalabad', 'Gujranwala',     'Gwadar',
  'Hyderabad',     'Multan',   'Peshawar',     'Quetta',    'Sialkot',
        'Wah',     'Sukkur']
Length: 17, dtype: str


Okay, great! We are now at 17 unique value count.

In [41]:
df_v3

,type,area,bedroom,bath,location_city,price
0,House,1 Kanal,7.0,6.0,Islamabad,PKR\n19 Crore
1,House,14.2 Marla,6.0,6.0,Islamabad,PKR\n6 Crore
2,House,1 Kanal,8.0,7.0,Islamabad,PKR\n7 Crore
3,House,8 Marla,4.0,6.0,Islamabad,PKR\n2.65 Crore
4,Flat,2.4 Marla,1.0,1.0,Islamabad,PKR\n40 Lakh
...,...,...,...,...,...,...
107269,Shop,50 Sqft,0.0,0.0,Sukkur,PKR 15.5 Lac
107270,Shop,360 Sqft,0.0,0.0,Sukkur,PKR 3 Crore
107271,Other,4 Kanal,0.0,0.0,Wah,PKR 5.5 Crore
107272,Shop,159 Sqft,0.0,0.0,Wah,PKR 38.4 Lac


---

You can see that we have 87567 rows, but our df is ending at index 108832. It's because we removed some values earlier and now we need to ***reset the indexes.***

In [42]:
df_v3 = df_v3.reset_index(drop=True)

In [43]:
df_v3

,type,area,bedroom,bath,location_city,price
0,House,1 Kanal,7.0,6.0,Islamabad,PKR\n19 Crore
1,House,14.2 Marla,6.0,6.0,Islamabad,PKR\n6 Crore
2,House,1 Kanal,8.0,7.0,Islamabad,PKR\n7 Crore
3,House,8 Marla,4.0,6.0,Islamabad,PKR\n2.65 Crore
4,Flat,2.4 Marla,1.0,1.0,Islamabad,PKR\n40 Lakh
...,...,...,...,...,...,...
87562,Shop,50 Sqft,0.0,0.0,Sukkur,PKR 15.5 Lac
87563,Shop,360 Sqft,0.0,0.0,Sukkur,PKR 3 Crore
87564,Other,4 Kanal,0.0,0.0,Wah,PKR 5.5 Crore
87565,Shop,159 Sqft,0.0,0.0,Wah,PKR 38.4 Lac


In [44]:
df_v3["type"]

0        House
1        House
2        House
3        House
4         Flat
         ...  
87562     Shop
87563     Shop
87564    Other
87565     Shop
87566     Flat
Name: type, Length: 87567, dtype: str

---

---

"**price**"

In [46]:
df_v3["price"].value_counts().tail(10)

price
PKR 85 Crore       1
PKR 1.35 Lac       1
PKR 79.75 Lac      1
PKR 22.18 Lac      1
PKR 24.75 Lac      1
PKR 1.4 Lac        1
PKR 16.25 Crore    1
PKR 70 Crore       1
PKR 11.94 Lac      1
PKR 18.27 Lac      1
Name: count, dtype: int64

SO the format seems to be same for all the rows : PKR Number Unit.
Let's remove PKR completely as we don't need it.

In [47]:
df_v3["price"]  = df_v3["price"].str.strip("PKR")
df_v3["price"]  = df_v3["price"].str.strip()
df_v3["price"].value_counts().head(10)

price
80 Lac        1646
1.5 Crore     1216
1.2 Crore     1093
1.4 Crore     1058
1.1 Crore     1047
2.5 Crore     1042
1 Crore        984
1.25 Crore     975
3.5 Crore      936
1.3 Crore      931
Name: count, dtype: int64

Okay, so the price column has different units as crore, lac, etc. Let's check how many are there in total.

In [48]:
print(df_v3["price"].str.split(" ").str[1:].str.join(" ").str.lower().value_counts())

price
crore       56184
lac         16686
lakh        14399
arab          205
               51
thousand       42
Name: count, dtype: int64


Okay, so there are mainly 5 different units. Actually 4, as lac/lakh are the same. I am gonna convert all these to complete numbers.

Now, let's convert them to numbers according to their units.

In [ ]:
def convert_to_rupees(price_str):
    match = str(price_str).split(" ")
    if len(match) < 2:
        return np.nan
    # I ran the it once and an error appeared : list index out of range. There must be some values without space.
    # Why str.len() checks the length of the list, if it has only one element(match[0]), it will return NaN at the position.

    amount = float(match[0])
    unit = match[1].lower()
    
    if "crore" in unit:
        return amount * 10000000
    elif "lakh" in unit or "lac" in unit:
        return amount * 100000
    elif "arab" in unit:
        return amount * 1000000000
    elif "thousand" in unit:
        return amount * 1000
    else:
        return np.nan
    
df_v3["price"] = df_v3["price"].apply(convert_to_rupees)

In [50]:
df_v3["price"].head()

0    190000000.0
1     60000000.0
2     70000000.0
3     26500000.0
4      4000000.0
Name: price, dtype: float64

In [51]:
df_v3["price"].isna().sum()

np.int64(51)

In [52]:
df_v3.info()

<class 'pandas.DataFrame'>
RangeIndex: 87567 entries, 0 to 87566
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           87567 non-null  str    
 1   area           87566 non-null  str    
 2   bedroom        87567 non-null  float64
 3   bath           87567 non-null  float64
 4   location_city  87567 non-null  str    
 5   price          87516 non-null  float64
dtypes: float64(3), str(3)
memory usage: 4.0 MB


Let's drop the null values.

In [53]:
df_v3 = df_v3.dropna()
df_v3 = df_v3.reset_index(drop=True)
df_v3.info()

<class 'pandas.DataFrame'>
RangeIndex: 87515 entries, 0 to 87514
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           87515 non-null  str    
 1   area           87515 non-null  str    
 2   bedroom        87515 non-null  float64
 3   bath           87515 non-null  float64
 4   location_city  87515 non-null  str    
 5   price          87515 non-null  float64
dtypes: float64(3), str(3)
memory usage: 4.0 MB


In [54]:
df_v3

,type,area,bedroom,bath,location_city,price
0,House,1 Kanal,7.0,6.0,Islamabad,190000000.0
1,House,14.2 Marla,6.0,6.0,Islamabad,60000000.0
2,House,1 Kanal,8.0,7.0,Islamabad,70000000.0
3,House,8 Marla,4.0,6.0,Islamabad,26500000.0
4,Flat,2.4 Marla,1.0,1.0,Islamabad,4000000.0
...,...,...,...,...,...,...
87510,Shop,50 Sqft,0.0,0.0,Sukkur,1550000.0
87511,Shop,360 Sqft,0.0,0.0,Sukkur,30000000.0
87512,Other,4 Kanal,0.0,0.0,Wah,55000000.0
87513,Shop,159 Sqft,0.0,0.0,Wah,3840000.0


---

"**area**"

In [55]:
df_v3["area"].head()

0       1 Kanal
1    14.2 Marla
2       1 Kanal
3       8 Marla
4     2.4 Marla
Name: area, dtype: str

It has the same format as price : Number Unit.

Let's see all the units.

In [56]:
print(df_v3["area"].str.split(" ").str[1:].str.join(" ").str.lower().value_counts().size)
df_v3["area"].str.split(" ").str[1:].str.join(" ").str.lower().value_counts()

28


area
marla                                            50200
kanal                                            16440
sqft                                              6306
sq. yd.                                           5902
sqyd                                              3689
yd²                                               2680
ft²                                               2247
sq. yd                                              19
                                                     6
sq. ft.                                              5
marla main double road                               2
marla corner                                         2
kanal corner                                         2
sqm                                                  1
sale 1 kanal                                         1
city sector b height location dua chowk              1
&amp; affordable living property 120 yd²             1
park 8 marla                                         1
3 mar

SO, what we are do now is that we are gonna convert all the main units to Marla. As it is the most commonly used in Pakistan.

We are gonna use only marla, kanal, sqft, sq. yd., sq. yd, sqyd, yd², ft², sq. ft., sqm, yard. Others are garbage values.

In [57]:
def convert_to_marla(area_str):
    match = str(area_str).strip().split(" ")
    if len(match) < 2:
        return np.nan
    
    try:
        number = float(match[0].replace(",", ""))
    except ValueError:
        return np.nan
    # There was a value error, so had to use exception handling
    
    unit = " ".join(match[1:]).lower()  #will join all the elements after match[0] by a space

    if "marla" in unit:
        return (number * 1)
    elif "kanal" in unit:
        return (number * 20)
    elif "sqft" in unit or "sq. ft." in unit:
        return (number / 272.25)
    elif "yd" in unit: #yd in unit catches yd², sqyd, and sq. yd. all at once
        return number / 30.25
    elif "ft²" in unit:
        return (number / 272.25)
    elif "sqm" in unit:
        return (number / 25.29)
    else:
        return np.nan
    
df_v3["area"] = df_v3["area"].apply(convert_to_marla)

In [58]:
df_v3["area"]

0        20.000000
1        14.200000
2        20.000000
3         8.000000
4         2.400000
           ...    
87510     0.183655
87511     1.322314
87512    80.000000
87513     0.584022
87514     1.836547
Name: area, Length: 87515, dtype: float64

In [59]:
df_v3 = df_v3.rename(columns={"area" : "area_in_marla"})
df_v3 = df_v3.dropna()
df_v3 = df_v3.reset_index(drop=True)
df_v3

,type,area_in_marla,bedroom,bath,location_city,price
0,House,20.000000,7.0,6.0,Islamabad,190000000.0
1,House,14.200000,6.0,6.0,Islamabad,60000000.0
2,House,20.000000,8.0,7.0,Islamabad,70000000.0
3,House,8.000000,4.0,6.0,Islamabad,26500000.0
4,Flat,2.400000,1.0,1.0,Islamabad,4000000.0
...,...,...,...,...,...,...
87494,Shop,0.183655,0.0,0.0,Sukkur,1550000.0
87495,Shop,1.322314,0.0,0.0,Sukkur,30000000.0
87496,Other,80.000000,0.0,0.0,Wah,55000000.0
87497,Shop,0.584022,0.0,0.0,Wah,3840000.0


---

# Analyzing the data

In [60]:
df_v4 = df_v3.copy()

In [61]:
df_v4.info()

<class 'pandas.DataFrame'>
RangeIndex: 87499 entries, 0 to 87498
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           87499 non-null  str    
 1   area_in_marla  87499 non-null  float64
 2   bedroom        87499 non-null  float64
 3   bath           87499 non-null  float64
 4   location_city  87499 non-null  str    
 5   price          87499 non-null  float64
dtypes: float64(4), str(2)
memory usage: 4.0 MB


In [62]:
df_v4.describe()

,area_in_marla,bedroom,bath,price
count,8.749900e+04,87499.000000,87499.000000,8.749900e+04
mean,6.235432e+02,1.863747,1.951119,3.846951e+07
std,5.088083e+04,2.308955,2.368515,2.450607e+08
min,0.000000e+00,0.000000,0.000000,2.340000e+03
25%,5.000000e+00,0.000000,0.000000,7.500000e+06
50%,7.000000e+00,0.000000,0.000000,1.550000e+07
75%,1.300000e+01,4.000000,4.000000,3.260000e+07
max,9.544372e+06,20.000000,10.000000,4.800000e+10


OK so there are some things that don't make sense at all. Very small prices and very large area.

In [63]:
# small prices
df_v4[df_v4["price"] < 100000].sort_values("price").head(10)

,type,area_in_marla,bedroom,bath,location_city,price
16700,Plot File,0.000000,0.0,0.0,Punjab,2340.0
20081,Other,0.000000,2.0,2.0,Punjab,3220.0
18399,Residential Plot,0.000000,0.0,0.0,Punjab,3430.0
8979,House,6998.000000,1.0,1.0,Other,6780.0
15220,Agricultural Land,920.000000,0.0,0.0,Other,10000.0
1298,House,1.000000,5.0,5.0,Other,10000.0
4762,House,1.000000,5.0,5.0,Other,10000.0
6429,House,0.826446,5.0,5.0,Other,12350.0
12425,Flat,2.000000,2.0,2.0,Other,25000.0
1330,House,5.000000,5.0,5.0,Other,25000.0


A 6998 marla house for 6k? Wow

Let's check how many houses has price less than 100k.

In [64]:
df_v4[df_v4["price"] < 100000].shape[0]

42

Only 42, let's drop them.

In [65]:
df_v4 = df_v4[df_v4["price"] >= 100000].reset_index(drop=True)

In [66]:
df_v4[df_v4["price"] < 100000].sort_values("price").head(10)

,type,area_in_marla,bedroom,bath,location_city,price


Now, let's check large area.

In [67]:
# suspiciously large areas
print(df_v4[df_v4["area_in_marla"] > 10000].shape[0])
df_v4[df_v4["area_in_marla"] > 10000].sort_values("area_in_marla", ascending=False).head(10)

320


,type,area_in_marla,bedroom,bath,location_city,price
28219,Residential Plot,9544372.0,0.0,0.0,Other,6.240000e+09
26424,Commercial Plot,9544372.0,0.0,0.0,Other,8.000000e+06
3070,House,4684680.0,5.0,5.0,Other,1.212000e+06
26409,Agricultural Land,2700000.0,0.0,0.0,Other,6.000000e+05
29943,Agricultural Land,2000000.0,0.0,0.0,Other,5.000000e+09
33384,Commercial Plot,2000000.0,0.0,0.0,Other,1.000000e+09
28102,Agricultural Land,1400000.0,0.0,0.0,Other,6.500000e+08
27185,Agricultural Land,1200000.0,0.0,0.0,Other,6.000000e+09
29408,Agricultural Land,800000.0,0.0,0.0,Other,7.500000e+08
25499,Agricultural Land,720000.0,0.0,0.0,Other,1.350000e+09


wtf is 9544372 marla?

In [68]:
df_v4[df_v4["area_in_marla"] > 10000]["type"].value_counts()

type
Agricultural Land    249
Other                 24
Residential Plot      20
Commercial Plot       13
House                  9
Building               3
Flat                   2
Name: count, dtype: int64

In [69]:
upper_limit = df_v4["area_in_marla"].quantile(0.99)
print(upper_limit)

1600.0


Quantile divides your data into 100 equal parts when you think of it as percentages. quantile(0.99) means: "find the value below which 99% of the data falls." So if it returns 1600, that means 99% of your properties have an area of 1600 Marla or less, and only the top 1% exceed it.

In [70]:
df_v4 = df_v4[df_v4["area_in_marla"] <= 1600].reset_index(drop=True)

In [71]:
df_v4.describe()

,area_in_marla,bedroom,bath,price
count,86621.000000,86621.000000,86621.000000,8.662100e+04
mean,18.197007,1.878632,1.966682,3.424267e+07
std,80.662079,2.313152,2.372468,1.236512e+08
min,0.000000,0.000000,0.000000,1.000000e+05
25%,5.000000,0.000000,0.000000,7.500000e+06
50%,7.000000,0.000000,0.000000,1.500000e+07
75%,12.000000,4.000000,4.000000,3.200000e+07
max,1600.000000,20.000000,10.000000,1.000000e+10


Properties with 0 area?...

In [72]:
df_v4 = df_v4[df_v4["area_in_marla"] > 0].reset_index(drop=True)

In [73]:
df_v4[df_v4["price"] >= 5000000000].sort_values("price", ascending=False).head(10)

,type,area_in_marla,bedroom,bath,location_city,price
62238,Commercial Plot,480.000000,0.0,0.0,Islamabad,1.000000e+10
32868,Building,220.000000,11.0,6.0,Islamabad,9.500000e+09
31175,Building,534.000000,6.0,6.0,Islamabad,9.000000e+09
31717,Building,160.000000,11.0,6.0,Lahore,9.000000e+09
31197,Building,480.000000,6.0,6.0,Islamabad,8.000000e+09
33002,Building,459.140496,6.0,6.0,Karachi,7.000000e+09
29670,Building,33.057851,6.0,6.0,Karachi,6.500000e+09
17127,Commercial Plot,100.000000,0.0,0.0,Islamabad,6.000000e+09
31023,Building,102.000000,5.0,5.0,Islamabad,5.500000e+09
6467,House,0.600000,5.0,5.0,Other,5.450000e+09


Everything else looks good except the house at 6467 index.

In [74]:
mask = df_v4[(df_v4["area_in_marla"] < 1) & (df_v4["price"] >= 50000)]
df_v4 = df_v4.drop(mask.index)
df_v4 = df_v4.reset_index(drop=True)
df_v4

,type,area_in_marla,bedroom,bath,location_city,price
0,House,20.000000,7.0,6.0,Islamabad,190000000.0
1,House,14.200000,6.0,6.0,Islamabad,60000000.0
2,House,20.000000,8.0,7.0,Islamabad,70000000.0
3,House,8.000000,4.0,6.0,Islamabad,26500000.0
4,Flat,2.400000,1.0,1.0,Islamabad,4000000.0
...,...,...,...,...,...,...
84972,Shop,3.305785,0.0,0.0,Sukkur,5900000.0
84973,Shop,2.571166,0.0,0.0,Sukkur,7500000.0
84974,Shop,1.322314,0.0,0.0,Sukkur,30000000.0
84975,Other,80.000000,0.0,0.0,Wah,55000000.0


In [75]:
df_v4.describe()

,area_in_marla,bedroom,bath,price
count,84977.000000,84977.000000,84977.000000,8.497700e+04
mean,18.537372,1.907045,1.994799,3.469514e+07
std,81.401120,2.317371,2.377936,1.233878e+08
min,1.000000,0.000000,0.000000,1.000000e+05
25%,5.000000,0.000000,0.000000,7.750000e+06
50%,7.000000,0.000000,0.000000,1.550000e+07
75%,12.000000,4.000000,4.000000,3.250000e+07
max,1600.000000,20.000000,10.000000,1.000000e+10


price max is 10 billion : that was the legitimate commercial plot in Islamabad

---

## Saving the dataframe as csv

In [76]:
df_v4.to_csv("data/cleaned_property_data.csv", index=False)